# Part 3: Advanced Modeling & Ensemble Methods

**House Prices: Advanced Regression Techniques** — Kaggle Competition

This notebook implements advanced feature engineering, skewness correction, model stacking,
and ensemble methods to improve upon the Part 2 baseline (0.13881 RMSE). Target: < 0.11 RMSE.

## 1. Imports & Configuration

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew
from scipy.special import boxcox1p
from scipy.optimize import minimize
from scipy.stats import boxcox_normmax

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR = os.path.join(BASE_DIR, "..", "data")
PLOT_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print("All imports OK")

All imports OK


## 2. Data Loading & Improved Missing Value Handling

Domain-specific imputation instead of blanket median fill.

In [2]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train shape: {train_raw.shape}")
print(f"Test  shape: {test_raw.shape}")

# Remove known GrLivArea outliers
train = train_raw[~((train_raw["GrLivArea"] > 4000) & (train_raw["SalePrice"] < 300000))].copy()
print(f"After outlier removal: {train.shape}")

# Log-transform target
y_train = np.log1p(train["SalePrice"])
train.drop("SalePrice", axis=1, inplace=True)

test_ids = test_raw["Id"]
test = test_raw.drop("Id", axis=1).copy()
train.drop("Id", axis=1, inplace=True)

n_train = len(y_train)
combined = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined shape: {combined.shape}")

Train shape: (1460, 81)
Test  shape: (1459, 80)
After outlier removal: (1458, 81)
Combined shape: (2917, 79)


In [3]:
# ── Domain-specific missing value imputation ──

# Features where NA means "not present" -> fill with "None" / 0
none_cols = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
             "BsmtFinType2", "FireplaceQu", "GarageType", "GarageFinish",
             "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature"]
for col in none_cols:
    combined[col] = combined[col].fillna("None")

zero_cols = ["GarageYrBlt", "GarageArea", "GarageCars",
             "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
             "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"]
for col in zero_cols:
    combined[col] = combined[col].fillna(0)

# LotFrontage: fill with median per Neighborhood (key improvement)
combined["LotFrontage"] = combined.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

# Mode-fill for categorical features
mode_cols = ["MSZoning", "Electrical", "KitchenQual", "Exterior1st", "Exterior2nd",
             "SaleType", "MasVnrType"]
for col in mode_cols:
    combined[col] = combined[col].fillna(combined[col].mode()[0])

# Functional: NA means Typ (per data description)
combined["Functional"] = combined["Functional"].fillna("Typ")

# Drop Utilities (near-zero variance - almost all AllPub)
combined.drop("Utilities", axis=1, inplace=True)

# Fill any remaining numeric NAs with 0, categorical with "None"
numeric_cols = combined.select_dtypes(include="number").columns
combined[numeric_cols] = combined[numeric_cols].fillna(0)
cat_cols = combined.select_dtypes(include="object").columns
combined[cat_cols] = combined[cat_cols].fillna("None")

print(f"Missing values remaining: {combined.isnull().sum().sum()}")
print(f"Shape after imputation: {combined.shape}")

Missing values remaining: 0
Shape after imputation: (2917, 78)


## 3. Feature Engineering

Create domain-informed features on the combined dataset before encoding.

In [4]:
# ── Area aggregates ──
combined["TotalSF"] = combined["TotalBsmtSF"] + combined["1stFlrSF"] + combined["2ndFlrSF"]
combined["TotalPorchSF"] = (combined["OpenPorchSF"] + combined["EnclosedPorch"] +
                             combined["3SsnPorch"] + combined["ScreenPorch"] +
                             combined["WoodDeckSF"])
combined["TotalBath"] = (combined["FullBath"] + 0.5 * combined["HalfBath"] +
                          combined["BsmtFullBath"] + 0.5 * combined["BsmtHalfBath"])
combined["TotalFinBsmtSF"] = combined["BsmtFinSF1"] + combined["BsmtFinSF2"]

# ── Binary indicators ──
combined["HasPool"] = (combined["PoolArea"] > 0).astype(int)
combined["HasGarage"] = (combined["GarageArea"] > 0).astype(int)
combined["HasBsmt"] = (combined["TotalBsmtSF"] > 0).astype(int)
combined["HasFireplace"] = (combined["Fireplaces"] > 0).astype(int)
combined["Has2ndFlr"] = (combined["2ndFlrSF"] > 0).astype(int)
combined["IsNew"] = (combined["YrSold"] == combined["YearBuilt"]).astype(int)
combined["IsRemodeled"] = (combined["YearRemodAdd"] != combined["YearBuilt"]).astype(int)

# ── Age features ──
combined["HouseAge"] = combined["YrSold"] - combined["YearBuilt"]
combined["RemodAge"] = combined["YrSold"] - combined["YearRemodAdd"]
combined["GarageAge"] = combined["YrSold"] - combined["GarageYrBlt"]
combined.loc[combined["GarageYrBlt"] == 0, "GarageAge"] = 0

# ── Quality interactions ──
combined["OverallScore"] = combined["OverallQual"] * combined["OverallCond"]

# ── Polynomial features ──
combined["OverallQual2"] = combined["OverallQual"] ** 2
combined["GrLivArea2"] = combined["GrLivArea"] ** 2
combined["TotalSF2"] = combined["TotalSF"] ** 2

print(f"Shape after feature engineering: {combined.shape}")
new_features = ["TotalSF", "TotalPorchSF", "TotalBath", "TotalFinBsmtSF",
                "HasPool", "HasGarage", "HasBsmt", "HasFireplace", "Has2ndFlr",
                "IsNew", "IsRemodeled", "HouseAge", "RemodAge", "GarageAge",
                "OverallScore", "OverallQual2", "GrLivArea2", "TotalSF2"]
print(f"New features created: {len(new_features)}")
for f in new_features:
    print(f"  {f}: mean={combined[f].mean():.2f}, std={combined[f].std():.2f}")

Shape after feature engineering: (2917, 96)
New features created: 18
  TotalSF: mean=2542.52, std=781.07
  TotalPorchSF: mean=182.70, std=159.76
  TotalBath: mean=2.22, std=0.81
  TotalFinBsmtSF: mean=488.46, std=466.58
  HasPool: mean=0.00, std=0.06
  HasGarage: mean=0.95, std=0.23
  HasBsmt: mean=0.97, std=0.16
  HasFireplace: mean=0.51, std=0.50
  Has2ndFlr: mean=0.43, std=0.49
  IsNew: mean=0.04, std=0.19
  IsRemodeled: mean=0.47, std=0.50
  HouseAge: mean=36.50, std=30.33
  RemodAge: mean=23.54, std=20.89
  GarageAge: mean=28.08, std=25.80
  OverallScore: mean=33.72, std=9.18
  OverallQual2: mean=39.02, std=17.72
  GrLivArea2: mean=2491591.48, std=1809436.52
  TotalSF2: mean=7074281.04, std=4790259.34


## 4. Skewness Correction (Box-Cox)

Apply Box-Cox transformation to highly skewed numeric features.

In [5]:
numeric_feats = combined.select_dtypes(include=["int64", "float64"]).columns
skewness = combined[numeric_feats].apply(lambda x: skew(x.dropna()))
high_skew = skewness[abs(skewness) > 0.75]
print(f"Features with |skewness| > 0.75: {len(high_skew)} out of {len(numeric_feats)}")

# Save before skewness for comparison plot
skew_before = skewness.copy()

# Apply Box-Cox1p transformation
for feat in high_skew.index:
    try:
        combined[feat] = boxcox1p(combined[feat], boxcox_normmax(combined[feat] + 1))
    except Exception:
        pass  # Skip if transformation fails

skew_after = combined[numeric_feats].apply(lambda x: skew(x.dropna()))

# Plot before vs after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(skew_before.values, bins=40, color="#e74c3c", alpha=0.7, edgecolor="black")
axes[0].set_title("Skewness Distribution BEFORE Box-Cox", fontsize=12)
axes[0].set_xlabel("Skewness")
axes[0].axvline(0.75, color="black", linestyle="--", label="|skew|=0.75")
axes[0].axvline(-0.75, color="black", linestyle="--")
axes[0].legend()

axes[1].hist(skew_after.values, bins=40, color="#2ecc71", alpha=0.7, edgecolor="black")
axes[1].set_title("Skewness Distribution AFTER Box-Cox", fontsize=12)
axes[1].set_xlabel("Skewness")
axes[1].axvline(0.75, color="black", linestyle="--", label="|skew|=0.75")
axes[1].axvline(-0.75, color="black", linestyle="--")
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "skewness_before_after.png"), dpi=150)
plt.show()

print(f"\nMean |skewness| before: {abs(skew_before).mean():.3f}")
print(f"Mean |skewness| after:  {abs(skew_after).mean():.3f}")

Features with |skewness| > 0.75: 31 out of 54



Mean |skewness| before: 3.066
Mean |skewness| after:  2.090


## 5. Encoding & Scaling

In [6]:
# ── Ordinal encode quality columns ──
quality_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
quality_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
                "HeatingQC", "KitchenQual", "FireplaceQu",
                "GarageQual", "GarageCond", "PoolQC"]

for col in quality_cols:
    combined[col] = combined[col].map(quality_map).fillna(0).astype(int)

# Additional ordinal mappings
bsmt_exposure_map = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
combined["BsmtExposure"] = combined["BsmtExposure"].map(bsmt_exposure_map).fillna(0).astype(int)

bsmt_fin_map = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}
for col in ["BsmtFinType1", "BsmtFinType2"]:
    combined[col] = combined[col].map(bsmt_fin_map).fillna(0).astype(int)

garage_fin_map = {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3}
combined["GarageFinish"] = combined["GarageFinish"].map(garage_fin_map).fillna(0).astype(int)

fence_map = {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4}
combined["Fence"] = combined["Fence"].map(fence_map).fillna(0).astype(int)

functional_map = {"Sal": 1, "Sev": 2, "Maj2": 3, "Maj1": 4, "Mod": 5, "Min2": 6, "Min1": 7, "Typ": 8}
combined["Functional"] = combined["Functional"].map(functional_map).fillna(8).astype(int)

# Quality interaction features (after ordinal encoding)
combined["ExterScore"] = combined["ExterQual"] * combined["ExterCond"]
combined["GarageScore"] = combined["GarageQual"] * combined["GarageCond"]
combined["KitchenScore"] = combined["KitchenQual"] * combined["KitchenAbvGr"]
combined["TotalQual"] = (combined["OverallQual"] + combined["ExterQual"] +
                          combined["BsmtQual"] + combined["KitchenQual"] +
                          combined["GarageQual"] + combined["HeatingQC"])

# ── One-hot encode remaining categoricals ──
remaining_cat = combined.select_dtypes(include="object").columns.tolist()
print(f"One-hot encoding {len(remaining_cat)} categorical columns...")
combined = pd.get_dummies(combined, columns=remaining_cat, drop_first=True)

print(f"Shape after encoding: {combined.shape}")

One-hot encoding 26 categorical columns...
Shape after encoding: (2917, 225)


In [7]:
# ── Split back & Scale ──
X_train_df = combined.iloc[:n_train].copy()
X_test_df  = combined.iloc[n_train:].copy()

# RobustScaler (less sensitive to outliers than StandardScaler)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_df),
    columns=X_train_df.columns,
    index=X_train_df.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_df),
    columns=X_test_df.columns,
    index=X_test_df.index
)

print(f"X_train: {X_train_scaled.shape}, X_test: {X_test_scaled.shape}")

X_train: (1458, 225), X_test: (1459, 225)


## 6. Feature Selection (Lasso)

Use Lasso to identify and remove noisy features.

In [8]:
# Fit Lasso for feature selection
lasso_fs = Lasso(alpha=0.0005, random_state=SEED, max_iter=10000)
lasso_fs.fit(X_train_scaled, y_train)

coef = pd.Series(lasso_fs.coef_, index=X_train_scaled.columns)
important_features = coef[abs(coef) > 1e-4].index.tolist()
dropped_features = coef[abs(coef) <= 1e-4].index.tolist()

print(f"Total features: {len(coef)}")
print(f"Selected features: {len(important_features)}")
print(f"Dropped features: {len(dropped_features)}")

# Apply feature selection
X_train_fs = X_train_scaled[important_features]
X_test_fs = X_test_scaled[important_features]

# Plot top 30 features by importance
top30 = coef[abs(coef) > 1e-4].abs().sort_values(ascending=False).head(30)
fig, ax = plt.subplots(figsize=(10, 8))
top30.plot(kind="barh", ax=ax, color="#3498db", edgecolor="grey")
ax.set_title("Top 30 Features by Lasso Coefficient Magnitude", fontsize=13)
ax.set_xlabel("|Coefficient|")
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "feature_importance.png"), dpi=150)
plt.show()

print(f"\nTop 10 features: {top30.index[:10].tolist()}")

Total features: 225
Selected features: 92
Dropped features: 133



Top 10 features: ['TotalSF', 'SaleType_New', 'Neighborhood_Crawfor', 'Neighborhood_StoneBr', 'TotalQual', 'HouseAge', 'SaleCondition_Normal', '2ndFlrSF', 'Neighborhood_NridgHt', 'OverallQual']


## 7. Individual Models with Cross-Validation

Train 7 models and compare their 5-fold CV RMSE.

In [9]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def rmse_cv(model, X, y):
    """Return array of RMSE from each fold."""
    scores = cross_val_score(model, X, y, cv=kf,
                             scoring="neg_mean_squared_error")
    return np.sqrt(-scores)

results = {}

# 1. Ridge
ridge = Ridge(alpha=10.0)
scores = rmse_cv(ridge, X_train_fs, y_train)
results["Ridge"] = scores
print(f"Ridge:            {scores.mean():.5f} (+/- {scores.std():.5f})")

# 2. Lasso
lasso = Lasso(alpha=0.0005, random_state=SEED, max_iter=10000)
scores = rmse_cv(lasso, X_train_fs, y_train)
results["Lasso"] = scores
print(f"Lasso:            {scores.mean():.5f} (+/- {scores.std():.5f})")

# 3. ElasticNet
enet = ElasticNet(alpha=0.0005, l1_ratio=0.9, random_state=SEED, max_iter=10000)
scores = rmse_cv(enet, X_train_fs, y_train)
results["ElasticNet"] = scores
print(f"ElasticNet:       {scores.mean():.5f} (+/- {scores.std():.5f})")

# 4. Kernel Ridge
kr = KernelRidge(alpha=0.6, kernel="polynomial", degree=2, coef0=2.5)
scores = rmse_cv(kr, X_train_fs, y_train)
results["KernelRidge"] = scores
print(f"KernelRidge:      {scores.mean():.5f} (+/- {scores.std():.5f})")

# 5. Gradient Boosting
gb = GradientBoostingRegressor(
    n_estimators=3000, learning_rate=0.05, max_depth=4,
    max_features="sqrt", min_samples_leaf=15, min_samples_split=10,
    loss="huber", random_state=SEED
)
scores = rmse_cv(gb, X_train_fs, y_train)
results["GradientBoosting"] = scores
print(f"GradientBoosting: {scores.mean():.5f} (+/- {scores.std():.5f})")

Ridge:            0.11019 (+/- 0.00578)


Lasso:            0.11156 (+/- 0.00630)


ElasticNet:       0.11139 (+/- 0.00621)
KernelRidge:      0.11775 (+/- 0.00878)


GradientBoosting: 0.11479 (+/- 0.00820)


In [10]:
# 6. XGBoost with Optuna tuning (50 trials)
print("Tuning XGBoost with Optuna (50 trials)...")

def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
        "gamma": trial.suggest_float("gamma", 0, 0.3),
        "random_state": SEED,
        "n_jobs": -1,
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X_train_fs, y_train, cv=kf,
                             scoring="neg_mean_squared_error")
    return np.sqrt(-scores).mean()

xgb_study = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

xgb_best_params = xgb_study.best_params
print(f"\nBest XGBoost params: {xgb_best_params}")

xgb_model = XGBRegressor(**xgb_best_params, random_state=SEED, n_jobs=-1)
scores = rmse_cv(xgb_model, X_train_fs, y_train)
results["XGBoost"] = scores
print(f"XGBoost:          {scores.mean():.5f} (+/- {scores.std():.5f})")

Tuning XGBoost with Optuna (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best XGBoost params: {'n_estimators': 1427, 'max_depth': 4, 'learning_rate': 0.016856468973453276, 'subsample': 0.6221580581897874, 'colsample_bytree': 0.642610469423013, 'min_child_weight': 4, 'reg_alpha': 0.0009151474290189521, 'reg_lambda': 1.7530567987317038, 'gamma': 0.02266015656749932}


XGBoost:          0.11410 (+/- 0.00783)


In [11]:
# 7. LightGBM with Optuna tuning (50 trials)
print("Tuning LightGBM with Optuna (50 trials)...")

def lgbm_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 0.9),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.9),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 7.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3.0),
        "num_leaves": trial.suggest_int("num_leaves", 20, 60),
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1,
    }
    model = LGBMRegressor(**params)
    scores = cross_val_score(model, X_train_fs, y_train, cv=kf,
                             scoring="neg_mean_squared_error")
    return np.sqrt(-scores).mean()

lgbm_study = optuna.create_study(direction="minimize",
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
lgbm_study.optimize(lgbm_objective, n_trials=50, show_progress_bar=True)

lgbm_best_params = lgbm_study.best_params
print(f"\nBest LightGBM params: {lgbm_best_params}")

lgbm_model = LGBMRegressor(**lgbm_best_params, random_state=SEED, n_jobs=-1, verbose=-1)
scores = rmse_cv(lgbm_model, X_train_fs, y_train)
results["LightGBM"] = scores
print(f"LightGBM:         {scores.mean():.5f} (+/- {scores.std():.5f})")

Tuning LightGBM with Optuna (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best LightGBM params: {'n_estimators': 1314, 'max_depth': 3, 'learning_rate': 0.015844931715922342, 'subsample': 0.716481202370179, 'colsample_bytree': 0.6675997745870428, 'min_child_weight': 6.5366524698494, 'reg_alpha': 1.6391253207142735e-06, 'reg_lambda': 1.4628814580041292, 'num_leaves': 24}


LightGBM:         0.11724 (+/- 0.00911)


In [12]:
# ── Model Comparison Plots ──
model_names = list(results.keys())
mean_scores = [results[m].mean() for m in model_names]
std_scores = [results[m].std() for m in model_names]

# Sort by mean score
sorted_idx = np.argsort(mean_scores)
model_names_sorted = [model_names[i] for i in sorted_idx]
mean_sorted = [mean_scores[i] for i in sorted_idx]
std_sorted = [std_scores[i] for i in sorted_idx]

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(model_names_sorted))]
bars = ax.barh(model_names_sorted, mean_sorted, xerr=std_sorted,
               color=colors, edgecolor="grey", capsize=3)
ax.set_xlabel("CV RMSE (log-scale SalePrice)", fontsize=11)
ax.set_title("Individual Model Comparison (5-Fold CV)", fontsize=13)
for bar, val in zip(bars, mean_sorted):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f"{val:.5f}", va="center", fontsize=9)
ax.set_xlim(left=min(mean_sorted) - 0.01)
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "model_comparison.png"), dpi=150)
plt.show()

# Boxplot of CV scores
fig, ax = plt.subplots(figsize=(10, 6))
cv_data = [results[m] for m in model_names_sorted]
bp = ax.boxplot(cv_data, labels=model_names_sorted, vert=False, patch_artist=True)
palette = sns.color_palette("Set2", len(model_names_sorted))
for patch, color in zip(bp["boxes"], palette):
    patch.set_facecolor(color)
ax.set_xlabel("CV RMSE", fontsize=11)
ax.set_title("Cross-Validation Score Distribution", fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "cv_scores_boxplot.png"), dpi=150)
plt.show()

# Print summary table
print("\n" + "="*55)
print(f"{'Model':<20} {'Mean RMSE':>12} {'Std':>10}")
print("="*55)
for m in model_names_sorted:
    print(f"{m:<20} {results[m].mean():>12.5f} {results[m].std():>10.5f}")
print("="*55)


Model                   Mean RMSE        Std
Ridge                     0.11019    0.00578
ElasticNet                0.11139    0.00621
Lasso                     0.11156    0.00630
XGBoost                   0.11410    0.00783
GradientBoosting          0.11479    0.00820
LightGBM                  0.11724    0.00911
KernelRidge               0.11775    0.00878


## 8. Ensemble / Stacking Methods

Three ensemble approaches:
- **A**: Weighted Average (optimize weights via scipy.optimize)
- **B**: Stacking with Meta-Learner (OOF predictions → Lasso meta-learner)
- **C**: Blending (stacked + XGB + LGBM)

In [13]:
# ── Prepare base models ──
base_models = {
    "Ridge": Ridge(alpha=10.0),
    "Lasso": Lasso(alpha=0.0005, random_state=SEED, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.0005, l1_ratio=0.9, random_state=SEED, max_iter=10000),
    "KernelRidge": KernelRidge(alpha=0.6, kernel="polynomial", degree=2, coef0=2.5),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=3000, learning_rate=0.05, max_depth=4,
        max_features="sqrt", min_samples_leaf=15, min_samples_split=10,
        loss="huber", random_state=SEED
    ),
    "XGBoost": XGBRegressor(**xgb_best_params, random_state=SEED, n_jobs=-1),
    "LightGBM": LGBMRegressor(**lgbm_best_params, random_state=SEED, n_jobs=-1, verbose=-1),
}

print(f"Base models: {list(base_models.keys())}")

Base models: ['Ridge', 'Lasso', 'ElasticNet', 'KernelRidge', 'GradientBoosting', 'XGBoost', 'LightGBM']


In [14]:
# ── Generate Out-of-Fold (OOF) predictions for stacking ──
X_np = X_train_fs.values
y_np = y_train.values
X_test_np = X_test_fs.values

oof_preds = np.zeros((len(y_np), len(base_models)))
test_preds = np.zeros((len(X_test_np), len(base_models)))

for i, (name, model) in enumerate(base_models.items()):
    print(f"Generating OOF for {name}...", end=" ")
    test_fold_preds = np.zeros((len(X_test_np), 5))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_np)):
        X_tr, X_val = X_np[train_idx], X_np[val_idx]
        y_tr, y_val = y_np[train_idx], y_np[val_idx]
        
        model_clone = type(model)(**model.get_params())
        model_clone.fit(X_tr, y_tr)
        
        oof_preds[val_idx, i] = model_clone.predict(X_val)
        test_fold_preds[:, fold] = model_clone.predict(X_test_np)
    
    test_preds[:, i] = test_fold_preds.mean(axis=1)
    oof_rmse = np.sqrt(mean_squared_error(y_np, oof_preds[:, i]))
    print(f"OOF RMSE: {oof_rmse:.5f}")

model_names_list = list(base_models.keys())
print("\nOOF predictions generated for all base models.")

Generating OOF for Ridge... OOF RMSE: 0.11035
Generating OOF for Lasso... 

OOF RMSE: 0.11175
Generating OOF for ElasticNet... 

OOF RMSE: 0.11157
Generating OOF for KernelRidge... OOF RMSE: 0.11807
Generating OOF for GradientBoosting... 

OOF RMSE: 0.11508
Generating OOF for XGBoost... 

OOF RMSE: 0.11437
Generating OOF for LightGBM... 

OOF RMSE: 0.11759

OOF predictions generated for all base models.


In [15]:
# ── Approach A: Weighted Average (optimize weights via scipy) ──
print("Approach A: Optimizing ensemble weights...")

def weighted_rmse(weights):
    """RMSE of weighted average of OOF predictions."""
    weights = np.abs(weights)
    weights = weights / weights.sum()
    blended = oof_preds @ weights
    return np.sqrt(mean_squared_error(y_np, blended))

n_models = len(base_models)
init_weights = np.ones(n_models) / n_models
bounds = [(0, 1)] * n_models
constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}

opt_result = minimize(weighted_rmse, init_weights, method="SLSQP",
                       bounds=bounds, constraints=constraints)
opt_weights = opt_result.x / opt_result.x.sum()

weighted_avg_rmse = weighted_rmse(opt_weights)
print(f"\nOptimal weights:")
for name, w in zip(model_names_list, opt_weights):
    print(f"  {name}: {w:.4f}")
print(f"Weighted Average OOF RMSE: {weighted_avg_rmse:.5f}")

# Generate weighted test predictions
weighted_test_pred = test_preds @ opt_weights

Approach A: Optimizing ensemble weights...

Optimal weights:
  Ridge: 0.4835
  Lasso: 0.0000
  ElasticNet: 0.0000
  KernelRidge: 0.1195
  GradientBoosting: 0.1525
  XGBoost: 0.2445
  LightGBM: 0.0000
Weighted Average OOF RMSE: 0.10709


In [16]:
# ── Approach B: Stacking with Meta-Learner ──
print("Approach B: Stacking with Lasso meta-learner...")

meta_model = Lasso(alpha=0.0005, random_state=SEED, max_iter=10000)

# CV for stacking
stacking_oof = np.zeros(len(y_np))
stacking_test_folds = np.zeros((len(X_test_np), 5))

for fold, (train_idx, val_idx) in enumerate(kf.split(oof_preds)):
    meta_clone = Lasso(alpha=0.0005, random_state=SEED, max_iter=10000)
    meta_clone.fit(oof_preds[train_idx], y_np[train_idx])
    stacking_oof[val_idx] = meta_clone.predict(oof_preds[val_idx])
    stacking_test_folds[:, fold] = meta_clone.predict(test_preds)

stacking_rmse = np.sqrt(mean_squared_error(y_np, stacking_oof))
print(f"Stacking OOF RMSE: {stacking_rmse:.5f}")

# Final stacking prediction
meta_model.fit(oof_preds, y_np)
stacking_test_pred = meta_model.predict(test_preds)
print(f"Meta-learner coefficients: {dict(zip(model_names_list, meta_model.coef_.round(4)))}")

Approach B: Stacking with Lasso meta-learner...
Stacking OOF RMSE: 0.10794
Meta-learner coefficients: {'Ridge': np.float64(0.4808), 'Lasso': np.float64(0.0), 'ElasticNet': np.float64(0.0), 'KernelRidge': np.float64(0.1183), 'GradientBoosting': np.float64(0.181), 'XGBoost': np.float64(0.2296), 'LightGBM': np.float64(0.0)}


In [17]:
# ── Approach C: Blending (stacked + XGB + LGBM) ──
print("Approach C: Blending stacked with XGB and LGBM...")

# Find XGB and LGBM indices
xgb_idx = model_names_list.index("XGBoost")
lgbm_idx = model_names_list.index("LightGBM")

blending_oof = 0.70 * stacking_oof + 0.15 * oof_preds[:, xgb_idx] + 0.15 * oof_preds[:, lgbm_idx]
blending_rmse = np.sqrt(mean_squared_error(y_np, blending_oof))
print(f"Blending OOF RMSE: {blending_rmse:.5f}")

blending_test_pred = 0.70 * stacking_test_pred + 0.15 * test_preds[:, xgb_idx] + 0.15 * test_preds[:, lgbm_idx]

# ── Compare all ensemble approaches ──
ensemble_results = {
    "A: Weighted Average": weighted_avg_rmse,
    "B: Stacking (Meta-Learner)": stacking_rmse,
    "C: Blending": blending_rmse,
}

print("\n" + "="*50)
print("Ensemble Method Comparison")
print("="*50)
for method, rmse in sorted(ensemble_results.items(), key=lambda x: x[1]):
    print(f"  {method}: {rmse:.5f}")

best_ensemble = min(ensemble_results, key=ensemble_results.get)
best_ensemble_rmse = ensemble_results[best_ensemble]
print(f"\nBest ensemble: {best_ensemble} (RMSE: {best_ensemble_rmse:.5f})")

# Ensemble comparison plot
fig, ax = plt.subplots(figsize=(8, 4))
methods = list(ensemble_results.keys())
rmses = list(ensemble_results.values())
sorted_idx = np.argsort(rmses)
methods_s = [methods[i] for i in sorted_idx]
rmses_s = [rmses[i] for i in sorted_idx]
colors = ["#2ecc71" if i == 0 else "#e67e22" for i in range(len(methods_s))]
bars = ax.barh(methods_s, rmses_s, color=colors, edgecolor="grey")
for bar, val in zip(bars, rmses_s):
    ax.text(val + 0.0003, bar.get_y() + bar.get_height()/2,
            f"{val:.5f}", va="center", fontsize=10)
ax.set_xlabel("OOF RMSE")
ax.set_title("Ensemble Method Comparison", fontsize=13)
ax.set_xlim(left=min(rmses_s) - 0.005)
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "ensemble_comparison.png"), dpi=150)
plt.show()

Approach C: Blending stacked with XGB and LGBM...
Blending OOF RMSE: 0.10858

Ensemble Method Comparison
  A: Weighted Average: 0.10709
  B: Stacking (Meta-Learner): 0.10794
  C: Blending: 0.10858

Best ensemble: A: Weighted Average (RMSE: 0.10709)


## 9. Residuals & Diagnostics

In [18]:
# Select best ensemble prediction for diagnostics
if "Weighted" in best_ensemble:
    best_oof = oof_preds @ opt_weights
    best_test_final = weighted_test_pred
elif "Stacking" in best_ensemble:
    best_oof = stacking_oof
    best_test_final = stacking_test_pred
else:
    best_oof = blending_oof
    best_test_final = blending_test_pred

residuals = y_np - best_oof

# Residuals plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(best_oof, residuals, alpha=0.4, s=15, c="#3498db", edgecolors="grey", linewidths=0.3)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted (log scale)")
axes[0].set_ylabel("Residual")
axes[0].set_title("Residuals vs Predicted", fontsize=12)

axes[1].hist(residuals, bins=40, color="#3498db", alpha=0.7, edgecolor="black")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Residual Distribution", fontsize=12)

plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "residuals_plot.png"), dpi=150)
plt.show()

# Prediction vs Actual
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_np, best_oof, alpha=0.4, s=15, c="#2ecc71", edgecolors="grey", linewidths=0.3)
min_val = min(y_np.min(), best_oof.min())
max_val = max(y_np.max(), best_oof.max())
ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual (log SalePrice)", fontsize=11)
ax.set_ylabel("Predicted (log SalePrice)", fontsize=11)
ax.set_title("Prediction vs Actual", fontsize=13)
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "prediction_vs_actual.png"), dpi=150)
plt.show()

print(f"Final OOF RMSE: {np.sqrt(mean_squared_error(y_np, best_oof)):.5f}")

Final OOF RMSE: 0.10709


## 10. Generate Kaggle Submission

In [19]:
# Reverse log-transform
preds_final = np.expm1(best_test_final)

# Clip any negative predictions
preds_final = np.maximum(preds_final, 0)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": preds_final})
submission_path = os.path.join(BASE_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Shape: {submission.shape}")
print(f"Id range: {submission['Id'].min()} - {submission['Id'].max()}")
print(f"Price range: ${submission['SalePrice'].min():,.0f} - ${submission['SalePrice'].max():,.0f}")
print(f"All prices > 0: {(submission['SalePrice'] > 0).all()}")
print(f"\nBest ensemble method: {best_ensemble}")
print(f"OOF RMSE: {best_ensemble_rmse:.5f}")
submission.head(10)

Submission saved to: /Users/ali.alsaifi/projects/personal-projects/HBKU/Spring-2026/Data Tools and Applications/midterm-project-excercise/part3_advanced_modeling/submission.csv
Shape: (1459, 2)
Id range: 1461 - 2919
Price range: $51,924 - $892,171
All prices > 0: True

Best ensemble method: A: Weighted Average
OOF RMSE: 0.10709


,Id,SalePrice
0,1461,117329.129111
1,1462,158647.987328
2,1463,179979.157832
3,1464,195446.411690
4,1465,190765.469623
5,1466,171366.703351
6,1467,180341.946443
7,1468,162967.684395
8,1469,191785.071369
9,1470,118550.106867


## Summary

**Key improvements over Part 2:**
1. Domain-specific imputation (LotFrontage by neighborhood, etc.)
2. 18 engineered features (area aggregates, quality interactions, age, polynomials)
3. Box-Cox skewness correction on 31 numeric features
4. RobustScaler instead of no scaling
5. Lasso-based feature selection (225 → 92 features)
6. 7 diverse models (linear + kernel + tree-based)
7. 3 ensemble approaches (weighted avg, stacking, blending)

**Results:**
- Best individual model: Ridge (CV RMSE: 0.11019)
- Best ensemble (Weighted Average): OOF RMSE: 0.10709
- **Part 2 Kaggle score:** 0.13881 RMSE → **Part 3 best Kaggle score:** 0.12217 RMSE
- Improvement: 0.01664 RMSE (12% relative improvement)

**Key finding:** Box-Cox skewness correction improved OOF RMSE but hurt Kaggle LB score.
The best LB score (0.12217) came from a simpler pipeline without Box-Cox transformation,
suggesting it introduced train/test distribution mismatch.